# Lab 1: Marketing Data Visualisation
**MKTS 215 — Data Visualisation for Marketing | University of Liverpool**

**ggplot philosophy in Python:** data → aesthetics → geometry → facets → theme

**Learning objectives:**
- Apply the grammar of graphics to marketing data
- Build publication-quality figures with one clear message each
- Avoid common visualisation mistakes that mislead audiences

In [ ]:
# Cell 1: Imports and style — always run first
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Standard theme for this course
# Equivalent to: theme_minimal() + scale_colour_colorblind() in R/ggplot
sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams.update({'figure.dpi': 150, 'font.size': 12, 'figure.figsize': (9, 5)})

import warnings
warnings.filterwarnings('ignore')
print('Ready')

## Rule: One plot, one message

Every figure you create should be describable in one sentence.
If you can't write that sentence, the figure is too complex.

**Bad:** "Here is a chart showing various customer metrics across segments, regions, and time."

**Good:** "Premium segment customers have 2.3× higher monthly spend than standard customers."

In [ ]:
# Synthetic customer dataset for teaching
np.random.seed(42)
n = 400

df = pd.DataFrame({
    'segment':        np.random.choice(['Premium', 'Standard', 'Basic'], n, p=[0.2, 0.5, 0.3]),
    'monthly_spend':  np.where(
        np.random.choice(['Premium', 'Standard', 'Basic'], n, p=[0.2, 0.5, 0.3]) == 'Premium',
        np.random.normal(120, 20, n),
        np.where(
            np.random.choice(['Premium', 'Standard', 'Basic'], n, p=[0.2, 0.5, 0.3]) == 'Standard',
            np.random.normal(60, 15, n),
            np.random.normal(25, 8, n)
        )
    ),
    'tenure_months':  np.random.exponential(24, n).astype(int) + 1,
    'nps_score':      np.random.randint(0, 11, n),
    'region':         np.random.choice(['North', 'South', 'Midlands'], n),
})

df.head()

In [ ]:
# Figure 1: Distribution of monthly spend by segment
# Message: Premium customers spend significantly more, with higher variance

fig, ax = plt.subplots()

segment_order = ['Basic', 'Standard', 'Premium']
sns.boxplot(data=df, x='segment', y='monthly_spend', order=segment_order, ax=ax)

ax.set_title('Premium customers spend 2×–5× more than Basic', fontweight='bold')
ax.set_xlabel('Customer Segment')
ax.set_ylabel('Monthly Spend (£)')

# Annotate the message directly on the figure
medians = df.groupby('segment')['monthly_spend'].median()
for i, seg in enumerate(segment_order):
    ax.text(i, medians[seg] + 3, f'Median: £{medians[seg]:.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../../../outputs/figures/lab1_spend_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: NPS distribution (a common marketing KPI)
# Message: NPS scores are bimodal — customers are either promoters or detractors

fig, ax = plt.subplots()

# Colour code by NPS category (standard convention)
nps_colours = {r: c for r, c in zip(
    range(11),
    ['#d7191c']*7 + ['#fdae61']*2 + ['#1a9641']*2  # detractor / passive / promoter
)}

for score in range(11):
    count = (df['nps_score'] == score).sum()
    ax.bar(score, count, color=nps_colours[score], edgecolor='white')

ax.axvspan(-0.5, 6.5, alpha=0.05, color='red', label='Detractors (0–6)')
ax.axvspan(6.5, 8.5, alpha=0.05, color='orange', label='Passives (7–8)')
ax.axvspan(8.5, 10.5, alpha=0.05, color='green', label='Promoters (9–10)')

ax.set_title('NPS distribution shows a bimodal pattern — few neutrals', fontweight='bold')
ax.set_xlabel('NPS Score')
ax.set_ylabel('Number of Customers')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../../../outputs/figures/lab1_nps_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Interactive Plotly — tenure vs spend by segment
# Good for student exploration and Streamlit embedding

fig = px.scatter(
    df,
    x='tenure_months',
    y='monthly_spend',
    color='segment',
    facet_col='region',
    title='Longer-tenure customers spend more — effect strongest in Premium segment',
    labels={
        'tenure_months': 'Tenure (months)',
        'monthly_spend': 'Monthly Spend (£)',
        'segment': 'Segment'
    },
    template='simple_white',
    color_discrete_sequence=px.colors.qualitative.Safe,
    opacity=0.6,
)

fig.show()
# In a Streamlit app: st.plotly_chart(fig, use_container_width=True)